# VectorDB, Chroma로 문서 저장·검색 실습
- numpy 로 만든 검색기는 직관에는 좋지만 청크 1만개를 넘어가면 매번 행렬 곱이 무거워짐
- **Chroma** 는 오픈 소스 임베디드 벡터 DB로, 노트북에 그대로 띄워서 쓸 수 있고 로컬 디스크에 영속화도 됨


## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 설치되어 있다면 실행하지 않아도 됩니다.


In [ ]:
# 필요한 라이브러리 설치
# uv add -qU langchain langchain-chroma langchain-openai langchain-text-splitters python-dotenv chromadb


## (2) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

## 2. VectorDB 핵심 개념

| 개념 | 의미 |
|---|---|
| Collection | 관련 문서 벡터를 담는 저장 단위 |
| Document | `page_content` 와 `metadata` 를 가진 LangChain 문서 객체 |
| Embedding function | 텍스트를 벡터로 바꾸는 함수 또는 모델 |
| Metadata | `source`, `section`, `owner` 같은 필터링·출처 정보 |
| ID | 문서를 갱신·삭제할 때 사용하는 고유 식별자 |
| Persist directory | vectorDB 데이터를 로컬 디스크에 저장하는 경로 |


## 3. 실습 문서 만들기

- 작은 사내 규정 문서를 `Document` 형태로 준비
- 실무에서는 PDF/Markdown/HTML 로더가 만든 문서도 같은 형태로 vectorDB 에 저장

In [ ]:
from langchain_core.documents import Document

# Document 객체로 추가 (텍스트 + 메타데이터)
policy_docs = [
    Document(
        page_content="신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.",
        metadata={"source": "hr_policy.md", "section": "보안교육", "owner": "HR", "version": "2026.06"},
    ),
    Document(
        page_content="법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.",
        metadata={"source": "expense_policy.md", "section": "경비처리", "owner": "Finance", "version": "2026.06"},
    ),
    Document(
        page_content="개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.",
        metadata={"source": "security_guide.md", "section": "개인정보", "owner": "Security", "version": "2026.06"},
    ),
    Document(
        page_content="장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.",
        metadata={"source": "dev_standards.md", "section": "장애보고", "owner": "Engineering", "version": "2026.06"},
    ),
    Document(
        page_content="재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.",
        metadata={"source": "hr_policy.md", "section": "재택근무", "owner": "HR", "version": "2026.06"},
    ),
    Document(
        page_content="회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회의는 조직장 승인이 필요하다.",
        metadata={"source": "office_guide.md", "section": "회의실", "owner": "Admin", "version": "2026.06"},
    ),
]

doc_ids = [f"policy-{i:02d}" for i in range(len(policy_docs))]

for doc_id, doc in zip(doc_ids, policy_docs):
    print(f"{doc_id} | {doc.metadata['section']} | {doc.page_content[:45]}...")


## 4. Chroma 첫 사용, 메모리 모드
- 메모리 모드는 노트북을 종료하면 사라지는 임시 저장소
- 빠른 실험과 수업용 데모에 적합함

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
print(f"저장 문서 수: {len()}")
print(f"Chroma count: {}")

## 5. 유사도 검색, `similarity_search`
- 질문을 임베딩한 뒤 collection 안의 문서 벡터와 비교해 가까운 문서를 반환


In [ ]:
results =

for doc in results:
    print(f"[{doc.metadata['section']}] {doc.page_content}")


## 6. 점수까지 보기, `similarity_search_with_score`
- Chroma 의 score 는 기본적으로 **distance** 인데, 값이 작을수록 더 가까움.
- 다른 retriever 의 relevance score 는 값이 클수록 관련도가 높은 경우가 있어 방향을 구분해야 함


In [ ]:
scored_results =

for doc, distance in scored_results:
    print(f"distance={distance:.3f} | [{doc.metadata['section']}] {doc.page_content[:80]}")


## 7. 메타데이터 필터 검색
- 문서 내용 검색과 별개로 `metadata` 조건을 걸 수 있음
- Chroma 의 `filter` 인자에 `{"메타필드": "값"}` 를 넘기면 메타데이터 조건 검색


In [ ]:
# Finance 담당 문서 안에서만 검색
finance_results =


for doc in finance_results:
    print(f"[{doc.metadata['owner']}/{doc.metadata['section']}] {doc.page_content}")


In [ ]:
# 여러 owner 를 한 번에 필터링
hr_or_admin_results =

for doc in hr_or_admin_results:
    print(f"[{doc.metadata['owner']}/{doc.metadata['section']}] {doc.page_content}")


## 8. 문서 추가, 갱신, 삭제
- 운영 환경에서는 문서가 계속 바뀜
- vectorDB 에도 문서 ID 기준의 추가·갱신·삭제 전략이 필요함


In [ ]:
# 추가
new_id = vectorstore.add_texts(
    texts=[" "],
    metadatas=[{"source": "expense_policy.md", "section": "교육비", "owner": "Finance", "version": "2026.06"}])
print(f"새 ID: {new_id}")

In [ ]:
# 확인
results =
for d in results:
    print(f"  {d.page_content}")

In [ ]:
# 삭제


# 확인
results =
for d in results:
    print(f"  {d.page_content}")

## 9. 영속화, 로컬 디스크에 저장

- `persist_directory` 를 지정하면 Chroma collection 이 로컬 폴더에 저장됨
- 노트북을 다시 열어도 같은 경로로 불러올 수 있음

In [ ]:
import shutil
from pathlib import Path

PERSIST_DIR = Path("./.chroma_company_policy")
shutil.rmtree(PERSIST_DIR, ignore_errors=True)

persisted_store = Chroma(
    collection_name="company_policy_disk",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIR),
)
persisted_store.add_documents(policy_docs, ids=doc_ids)
print("저장 count:", persisted_store._collection.count())


reloaded_store = Chroma(
    collection_name="company_policy_disk",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIR),
)
print("다시 로드한 count:", reloaded_store._collection.count())


## 10. Retriever 로 변환하기
- RAG 체인에 연결할 때는 `similarity_search()`를 직접 호출하기보다 `as_retriever()`로 변환해 두면 LCEL 체인에 끼우기 쉬움


In [ ]:
retriever =

retrieved_docs = retriever.invoke(" ")
for doc in retrieved_docs:
    print(f"[{doc.metadata['owner']}/{doc.metadata['section']}] {doc.page_content}")


In [ ]:
finance_retriever =

retrieved_docs = finance_retriever.invoke("")
for doc in retrieved_docs:
    print(f"[{doc.metadata['owner']}/{doc.metadata['section']}] {doc.page_content}")


## 11. 정리

- Chroma 는 로컬에서 바로 사용할 수 있는 vectorDB 입니다.
- `add_documents()` 로 문서와 메타데이터를 저장합니다.
- `similarity_search()` 는 유사 문서를 반환하고, `_with_score()` 는 distance 까지 반환합니다.
- `filter` 로 source, section, owner 같은 metadata 조건을 걸 수 있습니다.
- `persist_directory` 로 디스크에 저장하면 재실행 후에도 다시 로드할 수 있습니다.
- `as_retriever()` 로 바꾸면 RAG 체인에 연결하기 쉽습니다.


## [실습]

1. `k=1`, `k=3`, `k=5` 로 검색 결과를 비교합니다.
2. `filter={"section": "재택근무"}` 조건으로 재택근무 질문을 검색합니다.
3. `owner` 가 `HR` 또는 `Security` 인 문서만 검색하도록 `$in` 필터를 작성합니다.
4. 새 정책 문서 3개를 추가하고 검색 결과가 바뀌는지 확인합니다.
5. 05 차시의 `RAG_PROMPT`와 연결해 Chroma 기반 RAG 체인을 만듭니다.
